# Google ClusterData 2019 Traces

**Dataset:** https://github.com/google/cluster-data/blob/master/ClusterData2019.md

**Documentation:** `ClusterData2019_docs/Google cluster-usage traces v3.pdf`

This notebook fetches and explores the Google Borg cluster traces from May 2019.
The data is hosted on Google BigQuery and covers 8 Borg cells (a-h).

## 1. Setup & Authentication

### Prerequisites
1. A Google Cloud Platform project with billing enabled
2. BigQuery API enabled

### Create a Service Account Key
1. Go to **[IAM & Admin > Service Accounts](https://console.cloud.google.com/iam-admin/serviceaccounts)**
2. Click **"+ Create Service Account"**
3. Give it a name (e.g., `sembada-bigquery`), click **Create and Continue**
4. Grant the **"BigQuery User"** role, click **Continue**, then **Done**
5. Click on the newly created service account > **"Keys"** tab > **"Add Key" > "Create new key"**
6. Choose **JSON** and download the file
7. Place the downloaded JSON file in the project root as `credentials.json`

### Configure `.env` file
Create a `.env` file in the project root (copy from `.env.example`):
```env
GCP_PROJECT_ID=your-project-id-here
GOOGLE_APPLICATION_CREDENTIALS=credentials.json
```

In [ ]:
import os
import sys

# Add project root to path so we can import from functions/
sys.path.insert(0, os.path.abspath(".."))

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from functions.fetch_cluster_data import (
    get_bigquery_client,
    fetch_machine_capacity_sample,
    fetch_collection_events_sample,
    fetch_instance_usage_sample,
    fetch_cell_summary,
)

# Check configuration
PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "")
CREDENTIALS_PATH = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "")

print("--- Configuration Check ---")
print(f"GCP_PROJECT_ID: {PROJECT_ID or 'NOT SET'}")
print(f"GOOGLE_APPLICATION_CREDENTIALS: {CREDENTIALS_PATH or 'NOT SET'}")

if CREDENTIALS_PATH and os.path.exists(CREDENTIALS_PATH):
    print(f"Credentials file: FOUND ({CREDENTIALS_PATH})")
elif CREDENTIALS_PATH:
    print(f"Credentials file: NOT FOUND at {CREDENTIALS_PATH}")
else:
    print("Credentials file: NOT CONFIGURED")

if not PROJECT_ID:
    print("\nWARNING: GCP_PROJECT_ID not set in .env file.")
if not CREDENTIALS_PATH or not os.path.exists(CREDENTIALS_PATH):
    print("\nWARNING: Credentials file not found. Follow the setup steps above.")

## 2. Initialize BigQuery Client

In [ ]:
client = get_bigquery_client(project_id=PROJECT_ID)
print("BigQuery client initialized successfully.")

## 3. Test: Fetch Small Data Samples

These queries fetch small samples to prove we can connect and retrieve data from the BigQuery dataset.

### 3.1 Machine Capacity Sample

In [ ]:
result = fetch_machine_capacity_sample(client, cell="a", limit=10)
print(f"Table: {result.table} | Cell: {result.cell} | Rows: {result.row_count}")
print(f"\nQuery:\n{result.query}")
print(f"\nResults:")
display(result.dataframe)

### 3.2 Collection (Job) Events Sample

In [ ]:
result = fetch_collection_events_sample(client, cell="a", limit=10)
print(f"Table: {result.table} | Cell: {result.cell} | Rows: {result.row_count}")
print(f"\nResults:")
display(result.dataframe)

### 3.3 Instance Usage Sample

In [ ]:
result = fetch_instance_usage_sample(client, cell="a", limit=10)
print(f"Table: {result.table} | Cell: {result.cell} | Rows: {result.row_count}")
print(f"\nResults:")
display(result.dataframe)

### 3.4 Cell Summary Statistics

In [ ]:
result = fetch_cell_summary(client, cell="a")
print(f"Cell 'a' Summary:")
display(result.dataframe)